# 🛠️ Notebook 2: Car Rental — Implementation


## 🛠️ Setup

```bash
cd 07-object-oriented-design/car-rental
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> We build the system **three times**:
> 1. **v0 — bad:** one God class with flags and `if/elif` pricing. Works for
>    one car; falls apart fast.
> 2. **v1 — better:** split classes, polymorphic rates. Availability is
>    still naive.
> 3. **v2 — best:** overlapping-date availability, reservation state
>    machine, pricing strategy, payments, late fees, multi-branch search.
>
> Each version is self-contained — you can run any cell independently.


## 🔴 v0 — The naive "God class"

Beginner instinct: *"A rental is just a car with a flag saying whether
it is rented."* Let's write that and see where it hurts.


In [ ]:
# v0: everything crammed into one class. DO NOT COPY THIS STYLE.
class RentalV0:
    def __init__(self):
        # one dict per concern — classic smell
        self.cars = {}          # plate -> {"type": "car", "rented": False}
        self.customers = {}     # id   -> name

    def add_car(self, plate, kind):
        self.cars[plate] = {"type": kind, "rented": False}

    def rent(self, plate, customer_id, days):
        car = self.cars[plate]
        if car["rented"]:
            raise ValueError("already rented")
        # pricing by if/elif — every new vehicle type means editing this function
        if car["type"] == "car":     rate = 40
        elif car["type"] == "suv":   rate = 65
        elif car["type"] == "van":   rate = 80
        elif car["type"] == "truck": rate = 100
        else: raise ValueError("unknown type")
        car["rented"] = True
        return {"plate": plate, "customer": customer_id, "total": rate * days}

    def return_car(self, plate):
        self.cars[plate]["rented"] = False

shop = RentalV0()
shop.add_car("ABC-1", "car")
shop.add_car("XYZ-9", "suv")
print(shop.rent("ABC-1", customer_id=1, days=3))
print(shop.rent("XYZ-9", customer_id=2, days=1))
shop.return_car("ABC-1")


### What's wrong with v0?

1. **`rented: bool` can't answer "available next Friday?"** It only knows
   *right now*. You cannot book a car *ahead of time*.
2. **`if/elif` over vehicle type violates Open/Closed.** Adding a
   `Motorcycle` means editing `rent()` *and* every other method that
   switches on type. Forget one place → bug.
3. **No lifecycle.** There is no "confirmed but not picked up yet", no
   "was returned late", no way to cancel.
4. **One class owns everything** — storage, pricing, renting, returning.
   Testing one concern means dragging in all the others.

Let's fix #2 first.


## 🟡 v1 — Split classes + polymorphic rates

Instead of `if car["type"] == ...`, we make `Vehicle` an abstract class and
let each subtype answer `daily_rate()` for itself. This is the **Open/Closed
Principle**: open for extension (add `Motorcycle`), closed for modification
(no existing method has to change).


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

class Vehicle(ABC):
    def __init__(self, plate: str):
        self.plate = plate
        self.rented = False   # still naive — we fix this in v2
    @abstractmethod
    def daily_rate(self) -> float: ...
    @abstractmethod
    def seats(self) -> int: ...
    def __repr__(self): return f"{type(self).__name__}({self.plate})"

class Car(Vehicle):
    def daily_rate(self): return 40
    def seats(self): return 5

class SUV(Vehicle):
    def daily_rate(self): return 65
    def seats(self): return 7

# Adding a new type now costs ONE new class. No existing code changes.
class Motorcycle(Vehicle):
    def daily_rate(self): return 25
    def seats(self): return 2

@dataclass
class CustomerV1:
    id: int
    name: str

class RentalV1:
    def __init__(self, fleet):
        self.fleet = {v.plate: v for v in fleet}
    def rent(self, plate, customer, days):
        v = self.fleet[plate]
        if v.rented: raise ValueError("already rented")
        v.rented = True
        return {"plate": plate, "customer": customer.name, "total": v.daily_rate() * days}

shop = RentalV1([Car("ABC-1"), SUV("XYZ-9"), Motorcycle("MOT-1")])
print(shop.rent("ABC-1", CustomerV1(1, "Alice"), 3))
print(shop.rent("MOT-1", CustomerV1(2, "Bob"), 1))


### What's still wrong?

- Availability is still **a single boolean**. Two customers can't book the
  same car for two different, non-overlapping weeks.
- No reservation lifecycle → can't express "confirmed but not picked up".
- Pricing is one flat daily rate — no weekend surcharge, no weekly
  discount, no late fee.
- No payments.

On to v2 — the real design from Notebook 1.


## 🟢 v2 — The real design

We introduce:

- `ReservationState` enum + guarded transitions (the **State** pattern in
  10 lines).
- `overlaps(s, e)` on a reservation → availability is a *query* against
  the reservations list.
- `PricingPolicy` (the **Strategy** pattern) — weekend surcharge & weekly
  discount live in one place you can swap.
- `Payment` object with its own tiny state.
- `Branch` so we can have multiple locations.
- `RentalStore.drop_off(actual_date)` charges late fees.
- `VehicleFactory` (the **Factory** pattern) — the one place where a string
  from a CSV or an admin form becomes a real `Vehicle` subclass.
- A constructor **invariant** on `Reservation`: `end` may not precede `start`.


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import date, timedelta
from enum import Enum
from itertools import count

# ── Vehicles ────────────────────────────────────────────────────────────
class Vehicle(ABC):
    def __init__(self, plate: str):
        self.plate = plate
    @abstractmethod
    def daily_rate(self) -> float: ...
    @abstractmethod
    def seats(self) -> int: ...
    @property
    def category(self) -> str: return type(self).__name__
    def __repr__(self): return f"{self.category}({self.plate})"

class Car(Vehicle):
    def daily_rate(self): return 40
    def seats(self): return 5

class SUV(Vehicle):
    def daily_rate(self): return 65
    def seats(self): return 7

class Van(Vehicle):
    def daily_rate(self): return 80
    def seats(self): return 9

class Truck(Vehicle):
    def daily_rate(self): return 100
    def seats(self): return 3

@dataclass
class Customer:
    id: int
    name: str
    license_no: str

# ── Factory: turn fleet *data* into the right Vehicle subclass ──────────
# Fleet rows arrive as data — a CSV, an admin form, a JSON payload — so
# somewhere a string like "suv" has to become an `SUV` instance. Doing that
# with `if kind == "suv": ...` at every call site is the v0 mistake all over
# again, just moved. A registry keeps it in ONE place, and registering a new
# type is a dict entry rather than an edit to a conditional.
class VehicleFactory:
    _registry: dict[str, type[Vehicle]] = {
        "car": Car, "suv": SUV, "van": Van, "truck": Truck,
    }

    @classmethod
    def register(cls, kind: str, vehicle_cls: type[Vehicle]) -> None:
        if not issubclass(vehicle_cls, Vehicle):
            raise TypeError(f"{vehicle_cls.__name__} is not a Vehicle")
        cls._registry[kind.lower()] = vehicle_cls

    @classmethod
    def create(cls, kind: str, plate: str) -> Vehicle:
        try:
            return cls._registry[kind.lower()](plate)
        except KeyError:
            raise ValueError(
                f"unknown vehicle type {kind!r}; known: {sorted(cls._registry)}") from None

    @classmethod
    def build_fleet(cls, rows: list[tuple[str, str]]) -> list[Vehicle]:
        """rows are (kind, plate) pairs, e.g. straight out of a CSV."""
        return [cls.create(kind, plate) for kind, plate in rows]


In [ ]:
# ── Reservation state machine ──────────────────────────────────────────
class ReservationState(Enum):
    PENDING = "pending"
    CONFIRMED = "confirmed"
    ACTIVE = "active"
    RETURNED = "returned"
    CANCELLED = "cancelled"

_ids = count(1)

@dataclass
class Reservation:
    customer: Customer
    vehicle: Vehicle
    start: date
    end: date                    # inclusive
    id: int = field(default_factory=lambda: next(_ids))
    state: ReservationState = ReservationState.PENDING
    actual_return: date | None = None

    def __post_init__(self):
        # An invariant is a rule that must be true for the object to exist at
        # all, so it belongs in the constructor — not in whichever caller
        # happens to remember. Without this, `end < start` yields `days() == 0`
        # or a negative number, and the customer is charged nothing (or less).
        if self.end < self.start:
            raise ValueError(f"end {self.end} is before start {self.start}")

    # --- date math ------------------------------------------------------
    def overlaps(self, s: date, e: date) -> bool:
        # two ranges [a,b] and [c,d] overlap iff not (b < c or a > d)
        return not (e < self.start or s > self.end)

    def days(self) -> int:
        return (self.end - self.start).days + 1

    # --- state transitions (guarded) -----------------------------------
    def confirm(self):  self._require(ReservationState.PENDING);   self.state = ReservationState.CONFIRMED
    def pick_up(self):  self._require(ReservationState.CONFIRMED); self.state = ReservationState.ACTIVE
    def drop_off(self, actual: date | None = None):
        self._require(ReservationState.ACTIVE)
        self.actual_return = actual or self.end
        self.state = ReservationState.RETURNED
    def cancel(self):
        if self.state in (ReservationState.ACTIVE, ReservationState.RETURNED):
            raise ValueError("cannot cancel a picked-up reservation")
        self.state = ReservationState.CANCELLED

    def _require(self, s):
        if self.state != s:
            raise ValueError(f"illegal transition: expected {s.value}, got {self.state.value}")


In [ ]:
# ── Pricing (Strategy pattern) ─────────────────────────────────────────
class PricingPolicy(ABC):
    @abstractmethod
    def price(self, vehicle: Vehicle, start: date, end: date) -> float: ...

class FlatPricing(PricingPolicy):
    '''Simple: rate * number of days.'''
    def price(self, v, s, e):
        days = (e - s).days + 1
        return v.daily_rate() * days

class WeekendWeeklyPricing(PricingPolicy):
    '''Weekends cost 20% more; rentals of 7+ days get 10% off the total.'''
    WEEKEND_MULT = 1.2
    WEEKLY_DISCOUNT = 0.10
    def price(self, v, s, e):
        days = (e - s).days + 1
        total = 0.0
        for i in range(days):
            day = s + timedelta(days=i)
            rate = v.daily_rate()
            if day.weekday() >= 5:      # 5=Sat, 6=Sun
                rate *= self.WEEKEND_MULT
            total += rate
        if days >= 7:
            total *= (1 - self.WEEKLY_DISCOUNT)
        return round(total, 2)

# Try swapping the strategy — same vehicle, same dates, different total.
v = SUV("DEMO")
s, e = date(2025, 1, 3), date(2025, 1, 9)  # Fri–Thu, includes a weekend
print("flat           :", FlatPricing().price(v, s, e))
print("weekend+weekly :", WeekendWeeklyPricing().price(v, s, e))


In [ ]:
# ── Payments ───────────────────────────────────────────────────────────
class PaymentStatus(Enum):
    PENDING = "pending"
    PAID = "paid"
    REFUNDED = "refunded"

@dataclass
class Payment:
    amount: float
    method: str                               # "card", "cash", ...
    status: PaymentStatus = PaymentStatus.PENDING
    def charge(self):
        if self.status != PaymentStatus.PENDING:
            raise ValueError("already processed")
        self.status = PaymentStatus.PAID
    def refund(self):
        if self.status != PaymentStatus.PAID:
            raise ValueError("nothing to refund")
        self.status = PaymentStatus.REFUNDED


In [ ]:
# ── Branch + RentalStore ───────────────────────────────────────────────
class Branch:
    def __init__(self, name: str, fleet: list[Vehicle]):
        self.name = name
        self.fleet = fleet
        self.reservations: list[Reservation] = []

    def available(self, start: date, end: date, category: str | None = None) -> list[Vehicle]:
        busy = {r.vehicle.plate for r in self.reservations
                if r.state in (ReservationState.CONFIRMED, ReservationState.ACTIVE)
                and r.overlaps(start, end)}
        out = [v for v in self.fleet if v.plate not in busy]
        if category:
            out = [v for v in out if v.category == category]
        return out


class RentalStore:
    '''Orchestrates branches, pricing and payments.'''
    LATE_FEE_PENALTY = 25.0       # flat penalty per late return, on top of extra days

    def __init__(self, branches: list[Branch], pricing: PricingPolicy | None = None):
        self.branches = {b.name: b for b in branches}
        self.pricing = pricing or FlatPricing()
        self.payments: dict = {}   # reservation_id (or tuple) -> Payment

    def search(self, start: date, end: date, category: str | None = None):
        '''Search availability across every branch.'''
        return {name: b.available(start, end, category) for name, b in self.branches.items()}

    def book(self, branch: str, customer: Customer, vehicle: Vehicle,
             start: date, end: date, method: str = "card") -> Reservation:
        b = self.branches[branch]
        if vehicle not in b.available(start, end):
            raise ValueError("vehicle not available on those dates")
        r = Reservation(customer, vehicle, start, end)
        r.confirm()
        b.reservations.append(r)
        amount = self.pricing.price(vehicle, start, end)
        pay = Payment(amount=amount, method=method)
        pay.charge()
        self.payments[r.id] = pay
        return r

    def pick_up(self, r: Reservation):
        r.pick_up()

    def drop_off(self, r: Reservation, actual: date | None = None) -> float:
        '''Returns extra amount charged for late return (0 if on time).'''
        actual = actual or r.end
        extra = 0.0
        if actual > r.end:
            late_start = r.end + timedelta(days=1)
            extra = self.pricing.price(r.vehicle, late_start, actual) + self.LATE_FEE_PENALTY
            late_pay = Payment(amount=extra, method=self.payments[r.id].method)
            late_pay.charge()
            self.payments[(r.id, "late")] = late_pay
        r.drop_off(actual)
        return round(extra, 2)

    def cancel(self, r: Reservation):
        r.cancel()
        pay = self.payments.get(r.id)
        if pay and pay.status == PaymentStatus.PAID:
            pay.refund()


## 🧪 End-to-end demo

A small story that exercises the whole system: two branches, search across
both, a happy path, a collision (second booking fails), a cancellation
(with refund), and a late return (with fee).


In [ ]:
# Fleets come from data (imagine a CSV of the branch inventory), and the
# Factory decides which class each row becomes.
tlv = Branch("Tel Aviv", VehicleFactory.build_fleet([
    ("car", "TLV-CAR-1"), ("car", "TLV-CAR-2"), ("suv", "TLV-SUV-1")]))
hfa = Branch("Haifa", VehicleFactory.build_fleet([
    ("van", "HFA-VAN-1"), ("truck", "HFA-TRK-1")]))
print("TLV fleet built from data:", tlv.fleet)

# Extending the catalogue is a registration, not an edit to existing code.
class Motorcycle(Vehicle):
    def daily_rate(self): return 25
    def seats(self): return 2

VehicleFactory.register("motorcycle", Motorcycle)
print("after registering a new type:", VehicleFactory.create("motorcycle", "TLV-MOT-1"))
try:
    VehicleFactory.create("submarine", "SUB-1")
except ValueError as e:
    print("expected:", e)
print()
store = RentalStore([tlv, hfa], pricing=WeekendWeeklyPricing())

alice = Customer(1, "Alice", "DL-111")
bob   = Customer(2, "Bob",   "DL-222")
carol = Customer(3, "Carol", "DL-333")

# 1. Search — who has an SUV Fri–Sun in Tel Aviv?
fri, sun = date(2025, 1, 3), date(2025, 1, 5)
print("SUVs in TLV Fri–Sun:", store.search(fri, sun, category="SUV"))

# 2. Book the SUV for Alice
r1 = store.book("Tel Aviv", alice, tlv.fleet[2], fri, sun)
print(f"r1 booked: {r1}  paid ${store.payments[r1.id].amount}")

# 3. Collision: Bob tries to grab the same SUV for an overlapping range
try:
    store.book("Tel Aviv", bob, tlv.fleet[2], sun, sun + timedelta(days=2))
except ValueError as e:
    print("expected collision:", e)

# 4. Bob books a different car and then cancels → refund
r2 = store.book("Tel Aviv", bob, tlv.fleet[0], fri, sun)
print(f"r2 booked: {r2}")
store.cancel(r2)
print(f"r2 state: {r2.state.value}, payment: {store.payments[r2.id].status.value}")

# 5. Alice picks up, returns TWO DAYS LATE → late fee charged
store.pick_up(r1)
late = store.drop_off(r1, actual=r1.end + timedelta(days=2))
print(f"r1 late-fee charged: ${late}, final state: {r1.state.value}")

# 6. Carol rents a truck in Haifa for a full week → weekly discount kicks in
a, b = date(2025, 2, 3), date(2025, 2, 9)   # 7 days
r3 = store.book("Haifa", carol, hfa.fleet[1], a, b)
print(f"r3 total (7-day truck, with weekly 10% off): ${store.payments[r3.id].amount}")


## 🔍 Verify the design

Notebook 1 made five claims about this design. Here they are as executable
assertions — especially the date-overlap logic, which is the single easiest
place in this whole problem to be subtly, expensively wrong.

In [ ]:
from datetime import date as D

def store_with(pricing=None):
    b = Branch("T", VehicleFactory.build_fleet([("car", "T-1"), ("suv", "T-2")]))
    return b, RentalStore([b], pricing=pricing or FlatPricing())

alice = Customer(1, "Alice", "DL-1")
bob   = Customer(2, "Bob",   "DL-2")

def rejects(fn, *a, **kw):
    try:
        fn(*a, **kw)
    except ValueError:
        return True
    raise AssertionError("call should have been rejected")

# ── Overlap: the boundary cases are where booking systems bleed money ───
r = Reservation(alice, Car("X"), D(2025, 3, 10), D(2025, 3, 14))
assert r.days() == 5, "an inclusive range counts both end days"
assert r.overlaps(D(2025, 3, 10), D(2025, 3, 14))    # identical
assert r.overlaps(D(2025, 3, 12), D(2025, 3, 12))    # strictly inside
assert r.overlaps(D(2025, 3, 1),  D(2025, 3, 31))    # strictly around
assert r.overlaps(D(2025, 3, 14), D(2025, 3, 20))    # touches the last day
assert r.overlaps(D(2025, 3, 1),  D(2025, 3, 10))    # touches the first day
assert not r.overlaps(D(2025, 3, 15), D(2025, 3, 20))  # starts the day after
assert not r.overlaps(D(2025, 3, 1),  D(2025, 3, 9))   # ends the day before

# ── A reservation is a range, or it is not a reservation ───────────────
rejects(Reservation, alice, Car("X"), D(2025, 3, 14), D(2025, 3, 10))

# ── State machine: only the legal path is walkable ─────────────────────
r = Reservation(alice, Car("X"), D(2025, 3, 10), D(2025, 3, 14))
assert r.state is ReservationState.PENDING
rejects(r.pick_up)                      # cannot pick up before confirming
rejects(r.drop_off)                     # cannot return what was never collected
r.confirm(); rejects(r.confirm)         # no double confirm
r.pick_up()
rejects(r.cancel)                       # too late to cancel: the car is gone
r.drop_off(D(2025, 3, 14))
assert r.state is ReservationState.RETURNED
rejects(r.drop_off)                     # no double return

# ── Availability is a question about DATES, not a boolean flag ─────────
b, store = store_with()
car = b.fleet[0]
week1 = (D(2025, 4, 7), D(2025, 4, 11))
week2 = (D(2025, 4, 14), D(2025, 4, 18))
store.book("T", alice, car, *week1)
assert car not in b.available(*week1), "a booked car must disappear for those dates"
assert car in b.available(*week2), "…and stay bookable for a different week"
store.book("T", bob, car, *week2)      # the whole point of the redesign
rejects(store.book, "T", bob, car, D(2025, 4, 8), D(2025, 4, 9))
assert len(b.available(*week1, "SUV")) == 1, "category filter still works"

# ── A cancelled reservation frees the car; a refund follows the money ──
b, store = store_with()
car = b.fleet[0]
r = store.book("T", alice, car, *week1)
assert store.payments[r.id].status is PaymentStatus.PAID
store.cancel(r)
assert store.payments[r.id].status is PaymentStatus.REFUNDED
assert car in b.available(*week1), "cancelling must return the car to the pool"
r2 = store.book("T", bob, car, *week1)
assert r2.state is ReservationState.CONFIRMED

# ── Pricing is a swappable Strategy, and the rules really bite ─────────
suv = SUV("P")
fri_sun = (D(2025, 1, 3), D(2025, 1, 5))          # Fri, Sat, Sun
assert FlatPricing().price(suv, *fri_sun) == 65 * 3
assert WeekendWeeklyPricing().price(suv, *fri_sun) == round(65 + 65*1.2 + 65*1.2, 2)
week = (D(2025, 1, 6), D(2025, 1, 12))            # Mon–Sun, 7 days
assert WeekendWeeklyPricing().price(suv, *week) < FlatPricing().price(suv, *week), \
    "a 7-day rental should trigger the weekly discount"
# The store never names a policy class — it only calls .price().
assert "FlatPricing" not in RentalStore.book.__code__.co_names
assert "price" in RentalStore.book.__code__.co_names

# ── Late returns cost extra, on top of the original payment ────────────
b, store = store_with()
car = b.fleet[0]
r = store.book("T", alice, car, D(2025, 5, 5), D(2025, 5, 6))
store.pick_up(r)
assert store.drop_off(r, actual=D(2025, 5, 6)) == 0.0, "on time is free"
r2 = store.book("T", bob, b.fleet[1], D(2025, 6, 2), D(2025, 6, 3))
store.pick_up(r2)
late = store.drop_off(r2, actual=D(2025, 6, 5))   # two days over
assert late == round(65 * 2 + RentalStore.LATE_FEE_PENALTY, 2), late
assert store.payments[(r2.id, "late")].status is PaymentStatus.PAID
assert store.payments[r2.id].amount != late, "the late charge is a separate payment"

# ── Payment guards its own tiny state machine ──────────────────────────
pay = Payment(10.0, "card")
rejects(pay.refund)                      # nothing to refund yet
pay.charge(); rejects(pay.charge)        # no double charge
pay.refund(); rejects(pay.refund)        # no double refund

# ── Factory: data in, the right class out ──────────────────────────────
assert isinstance(VehicleFactory.create("SUV", "Q-1"), SUV)     # case-insensitive
assert [type(v).__name__ for v in VehicleFactory.build_fleet(
            [("car", "a"), ("truck", "b")])] == ["Car", "Truck"]
rejects(VehicleFactory.create, "hovercraft", "H-1")
try:
    VehicleFactory.register("nonsense", str); raise AssertionError("accepted a non-Vehicle")
except TypeError:
    pass

print("✅ overlap maths, state machine, availability, pricing, late fees and Factory verified")

## 🧠 Recap — what each version taught us

| Pain point in v0/v1 | Fix in v2 | Pattern |
|---|---|---|
| `is_available` boolean | `overlaps(s,e)` query | — |
| `if/elif` on vehicle type | `Vehicle` subclasses | Polymorphism |
| No lifecycle → illegal moves possible | `ReservationState` + guards | State |
| Pricing hard-coded | `PricingPolicy` injected | Strategy |
| One class does everything | Branch / Store / Payment / Pricing | Single-Responsibility |
| `if kind == "suv"` when loading fleet data | `VehicleFactory` registry | Factory |
| Nothing stopped `end < start` | invariant in `Reservation.__post_init__` | Design by contract |


## 🚀 Try it yourself

1. **Add a loyalty tier.** Create `LoyaltyPricing(PricingPolicy)` that
   gives a 15% discount to customers with `vip=True`. You'll only edit
   pricing.

2. **GPS & child-seat add-ons.** Let `book()` accept a list of `Addon`
   objects, each with a `price_per_day`. Add it to the total.

3. **One-way rental.** Allow `drop_off` at a *different* branch. When
   that happens, charge a fixed relocation fee and move the vehicle to
   the drop-off branch's fleet.

4. **Persistence.** Replace `branch.reservations` (a list) with a SQLite
   table. Notice how `overlaps()` becomes a SQL `WHERE` clause.

5. **Concurrency.** What happens if two `book()` calls run at the same
   time for the same car and dates? Sketch how you'd prevent it (hint:
   unique constraint on `(vehicle_id, date)` in a bookings-per-day
   table).
